"""
아래 데이터는 예시인데 이런 식으로 결과 값이 나오도록 파일 하나 csv 파일 만들어줘

period	긍정	중립	부정
before	4.580335731414868	57.625899280575545	37.79376498800959
after	2.9305789094403076	62.81527744415085	34.25414364640884
"""

## 1단계: 필수 패키지 설치 및 감정 분석 파이프라인 로드

Hugging Face의 `transformers`와 감정 분석용 KLUE-BERT v2 모델을 로드하기 위해 필요한 패키지를 로드합니다.
이 모델은 특정 라이브러리 버전 버그가 있어 아래 지정된 버전의 `transformers` 및 `huggingface-hub`를 사용하여 구동하는 것을 권장합니다.

In [45]:
# 만약 패키지가 설치되지 않은 경우 아래 주석을 해제하고 실행해 주세요.
!pip install transformers==4.44.2 huggingface-hub==0.24.7 torch pandas


In [46]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 1. 토크나이저 및 모델 로드
model_name = "dlckdfuf141/korean-emotion-kluebert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 2. 감정 분류 파이프라인 생성 (기본 CPU 사용)
emotion_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=-1)

# 3. 파이프라인 테스트
test_text = "오늘 하루 너무 즐거웠어!"
result = emotion_classifier(test_text)
print("테스트 결과:", result)

테스트 결과: [{'label': 5, 'score': 0.9999791383743286}]


## 2단계: 데이터셋 로드 및 감정 분석 수행 (일괄 처리)

`News_Scraping.csv` 파일을 읽어서 각 기사 제목의 감정을 분석한 뒤, `감정` 열에 예측한 라벨을 한글로 매핑하여 저장하고, `수치` 열에 감정 스코어(확률)를 기입하여 덮어쓰기 저장합니다.


In [47]:
import pandas as pd
import os

csv_file_path = "data/News_Scraping_retouch.csv"

# 1. CSV 데이터 로드
if os.path.exists(csv_file_path):
    df = pd.read_csv(csv_file_path)
    print(f"성공적으로 데이터를 로드했습니다. 총 기사 수: {len(df)}")
else:
    raise FileNotFoundError(f"{csv_file_path} 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")

# 2. 모델 라벨 -> 한글 감정 명칭 변환 딕셔너리 정의
emotion_map = {
    "LABEL_0": "공포", "0": "공포", 0: "공포",
    "LABEL_1": "놀람", "1": "놀람", 1: "놀람",
    "LABEL_2": "분노", "2": "분노", 2: "분노",
    "LABEL_3": "슬픔", "3": "슬픔", 3: "슬픔",
    "LABEL_4": "중립", "4": "중립", 4: "중립",
    "LABEL_5": "행복", "5": "행복", 5: "행복",
    "LABEL_6": "혐오", "6": "혐오", 6: "혐오"
}

# 3. 뉴스 기사 제목 감정 분석 일괄 처리
emotions = []
scores = []

print("뉴스 기사 제목 감정 분석 시작...")
for idx, row in df.iterrows():
    title = row['기사제목']
    if pd.isna(title) or not str(title).strip():
        emotions.append(None)
        scores.append(None)
        continue
        
    try:
        # 감정 분석 예측
        prediction = emotion_classifier(str(title))[0]
        raw_label = prediction['label']
        
        # 한글 감정 라벨로 변환 (매핑 딕셔너리에 없으면 원래 라벨 유지)
        mapped_label = emotion_map.get(raw_label, raw_label)
        
        emotions.append(mapped_label)
        scores.append(prediction['score'])
    except Exception as e:
        print(f"Error at index {idx} ('{title}'): {e}")
        emotions.append(None)
        scores.append(None)

    # 진행 상태 출력
    if (idx + 1) % 100 == 0:
        print(f"진행 상황: {idx + 1}/{len(df)} 완료")

# 4. 데이터프레임 업데이트
df['감정'] = emotions
df['수치'] = scores

# 5. 분석 결과 파일 덮어쓰기 저장 (UTF-8-SIG 인코딩 사용으로 한글 깨짐 방지)
df.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
print("감정 분석 완료 및 News_Scraping.csv 파일에 저장되었습니다!")


성공적으로 데이터를 로드했습니다. 총 기사 수: 11229
뉴스 기사 제목 감정 분석 시작...
진행 상황: 100/11229 완료
진행 상황: 200/11229 완료
진행 상황: 300/11229 완료
진행 상황: 400/11229 완료
진행 상황: 500/11229 완료
진행 상황: 600/11229 완료
진행 상황: 700/11229 완료
진행 상황: 800/11229 완료
진행 상황: 900/11229 완료
진행 상황: 1000/11229 완료
진행 상황: 1100/11229 완료
진행 상황: 1200/11229 완료
진행 상황: 1300/11229 완료
진행 상황: 1400/11229 완료
진행 상황: 1500/11229 완료
진행 상황: 1600/11229 완료
진행 상황: 1700/11229 완료
진행 상황: 1800/11229 완료
진행 상황: 1900/11229 완료
진행 상황: 2000/11229 완료
진행 상황: 2100/11229 완료
진행 상황: 2200/11229 완료
진행 상황: 2300/11229 완료
진행 상황: 2400/11229 완료
진행 상황: 2500/11229 완료
진행 상황: 2600/11229 완료
진행 상황: 2700/11229 완료
진행 상황: 2800/11229 완료
진행 상황: 2900/11229 완료
진행 상황: 3000/11229 완료
진행 상황: 3100/11229 완료
진행 상황: 3200/11229 완료
진행 상황: 3300/11229 완료
진행 상황: 3400/11229 완료
진행 상황: 3500/11229 완료
진행 상황: 3600/11229 완료
진행 상황: 3700/11229 완료
진행 상황: 3800/11229 완료
진행 상황: 3900/11229 완료
진행 상황: 4000/11229 완료
진행 상황: 4100/11229 완료
진행 상황: 4200/11229 완료
진행 상황: 4300/11229 완료
진행 상황: 4400/11229 완료
진행 상황: 4500/11229 완료
진행 상황: 460

# --- 3.1. 시기별 통계를 계산하는 개별 하위 함수 정의 (프린트문 없음) ---

def analyze_pre_implementation(df_policy):
    df_sub = df_policy[df_policy['시기'] == '시행전']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats

def analyze_on_implementation(df_policy):
    df_sub = df_policy[df_policy['시기'] == '시행일']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats

def analyze_early_reaction(df_policy):
    df_sub = df_policy[df_policy['시기'] == '초기반응']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats

def analyze_perceived_reaction(df_policy):
    df_sub = df_policy[df_policy['시기'] == '체감반응']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats


# --- 3.2. 시기별 통계를 종합하여 딕셔너리로 반환하는 함수 ---

def analyze_policy_emotions(df, policy_name=None):
    """
    더 이상 '정책' 필터링을 하지 않고, 전달받은 데이터프레임 전체를 대상으로 통계를 냅니다.
    """
    df_policy = df # 정책 컬럼이 없으므로 필터링 배제
    if df_policy.empty:
        return {}
        
    period_functions = {
        '시행전': analyze_pre_implementation,
        '시행일': analyze_on_implementation,
        '초기반응': analyze_early_reaction,
        '체감반응': analyze_perceived_reaction
    }
    
    results = {}
    for period_name, func in period_functions.items():
        stats = func(df_policy)
        if stats is not None:
            stats['평균수치'] = stats['평균수치'].round(4)
            results[period_name] = stats
        else:
            results[period_name] = None
            
    return results


# --- 3.3. 시기별/감정별 기사 제목 리스트 추출 함수 ---

def get_emotion_titles_by_period(df_policy, period_name):
    df_period = df_policy[df_policy['시기'] == period_name]
    if df_period.empty:
        return {}
    grouped = df_period.groupby('감정')['기사제목'].apply(list).to_dict()
    return grouped

def get_policy_emotions_titles(df, policy_name=None):
    df_policy = df # 정책 필터링 배제
    if df_policy.empty:
        return {}
        
    periods = ['시행전', '시행일', '초기반응', '체감반응']
    results = {}
    for period in periods:
        results[period] = get_emotion_titles_by_period(df_policy, period)
    return results

In [48]:
# --- 3.1. 시기별 통계를 계산하는 개별 하위 함수 정의 (프린트문 없음) ---

def analyze_pre_implementation(df_policy):
    df_sub = df_policy[df_policy['시기'] == '시행전']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats

def analyze_on_implementation(df_policy):
    df_sub = df_policy[df_policy['시기'] == '시행일']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats

def analyze_early_reaction(df_policy):
    df_sub = df_policy[df_policy['시기'] == '초기반응']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats

def analyze_perceived_reaction(df_policy):
    df_sub = df_policy[df_policy['시기'] == '체감반응']
    if df_sub.empty:
        return None
    stats = df_sub.groupby('감정')['수치'].agg(['count', 'mean']).rename(columns={'count': '기사수', 'mean': '평균수치'})
    return stats


# --- 3.2. 시기별 통계를 종합하여 딕셔너리로 반환하는 함수 ---

def analyze_policy_emotions(df, policy_name=None):
    """
    정책 컬럼 제거에 대응하여 필터링 없이 전체 데이터를 대상으로 통계를 도출합니다.
    """
    df_policy = df
    if df_policy.empty:
        return {}
        
    period_functions = {
        '시행전': analyze_pre_implementation,
        '시행일': analyze_on_implementation,
        '초기반응': analyze_early_reaction,
        '체감반응': analyze_perceived_reaction
    }
    
    results = {}
    for period_name, func in period_functions.items():
        stats = func(df_policy)
        if stats is not None:
            stats['평균수치'] = stats['평균수치'].round(4)
            results[period_name] = stats
        else:
            results[period_name] = None
            
    return results


# --- 3.3. 시기별/감정별 기사 제목 리스트 추출 함수 ---

def get_emotion_titles_by_period(df_policy, period_name):
    df_period = df_policy[df_policy['시기'] == period_name]
    if df_period.empty:
        return {}
    grouped = df_period.groupby('감정')['기사제목'].apply(list).to_dict()
    return grouped

def get_policy_emotions_titles(df, policy_name=None):
    df_policy = df
    if df_policy.empty:
        return {}
        
    periods = ['시행전', '시행일', '초기반응', '체감반응']
    results = {}
    for period in periods:
        results[period] = get_emotion_titles_by_period(df_policy, period)
    return results

policy_name = "6·27 가계부채 관리 강화방안"
df_policy_cleaned = df # 정책 필터링 배제

print("============================================================")
print(f" [부동산 정책 통계 출력] 정책명: {policy_name}")
print("============================================================")

stats_dict = analyze_policy_emotions(df_policy_cleaned)
for period, stats in stats_dict.items():
    print(f"\n>>> 시기: [{period}]")
    if stats is not None:
        print(stats)
    else:
        print("    데이터 없음")

print("\n============================================================")
print(f" [부동산 정책 감정별 기사 제목 출력] 정책명: {policy_name}")
print("============================================================")

titles_dict = get_policy_emotions_titles(df_policy_cleaned)
for period, emotions in titles_dict.items():
    print(f"\n>>> 시기: [{period}]")
    if emotions:
        for emotion, titles in emotions.items():
            print(f"  * 감정: {emotion} (총 {len(titles)}건)")
            for t in titles[:3]:
                print(f"    - {t}")
            if len(titles) > 3:
                print(f"    - ...외 {len(titles)-3}건")
    else:
        print("    데이터 없음")

In [49]:
policy_name = "6·27 가계부채 관리 강화방안"
df_policy_cleaned = df_analyzed if 'df_analyzed' in globals() else df # 동적 매핑

print("============================================================")
print(f" [부동산 정책 통계 출력] 정책명: {policy_name}")
print("============================================================")

stats_dict = analyze_policy_emotions(df_policy_cleaned)
for period, stats in stats_dict.items():
    print(f"\n>>> 시기: [{period}]")
    if stats is not None:
        print(stats)
    else:
        print("    데이터 없음")

print("\n============================================================")
print(f" [부동산 정책 감정별 기사 제목 출력] 정책명: {policy_name}")
print("============================================================")

titles_dict = get_policy_emotions_titles(df_policy_cleaned)
for period, emotions in titles_dict.items():
    print(f"\n>>> 시기: [{period}]")
    if emotions:
        for emotion, titles in emotions.items():
            print(f"  * 감정: {emotion} (총 {len(titles)}건)")
            for t in titles[:3]:
                print(f"    - {t}")
            if len(titles) > 3:
                print(f"    - ...외 {len(titles)-3}건")
    else:
        print("    데이터 없음")

 [부동산 정책 통계 출력] 정책명: 6·27 가계부채 관리 강화방안

>>> 시기: [시행전]
     기사수    평균수치
감정              
공포    87  0.9054
놀람   360  0.9568
분노    31  0.9069
슬픔   173  0.8960
중립  1897  0.9816
행복   199  0.9368
혐오   101  0.9006

>>> 시기: [시행일]
    기사수    평균수치
감정             
공포    4  0.7742
놀람   12  0.9506
분노    2  0.9892
슬픔   12  0.9617
중립   41  0.9736
행복    5  0.9372
혐오    9  0.9084

>>> 시기: [초기반응]
     기사수    평균수치
감정              
공포    82  0.9069
놀람   277  0.9478
분노    32  0.9101
슬픔   209  0.9238
중립  1975  0.9761
행복   166  0.9302
혐오    99  0.9439

>>> 시기: [체감반응]
     기사수    평균수치
감정              
공포   129  0.9280
놀람   524  0.9422
분노   105  0.9298
슬픔   355  0.8923
중립  3955  0.9793
행복   309  0.9294
혐오   264  0.9425

 [부동산 정책 감정별 기사 제목 출력] 정책명: 6·27 가계부채 관리 강화방안

>>> 시기: [시행전]
  * 감정: 공포 (총 87건)
    - 세운상가 재개발 예정구역 큰 불, 사업 차질? 서울시 "지연 없을 듯"
    - 수도권도 서울만 0.31% 국지적 상승…경기·인천 0.08% 떨어져
    - 착공 34%, 분양 41% 급감...‘주택 공급 절벽' 우려 커진다
    - ...외 84건
  * 감정: 놀람 (총 360건)
    - 서울 아파트값 17주 연속 올라…강남3구·양천구↑
    - 아파트 스토

# Streamlit 화면 예시
    import streamlit as st

    # 통계 데이터 렌더링
    stats_dict = analyze_policy_emotions(df, "4·1 부동산 대책")
    st.dataframe(stats_dict['초기반응'])

    # 감정별 기사제목 렌더링
    titles_dict = get_policy_emotions_titles(df, "4·1 부동산 대책")
    st.write(titles_dict['초기반응']['분노'])

## 5단계: 시행일 기준 감정 증감 분석 (부정 감정 통합 및 표본 크기 보완)

공포, 분노, 슬픔, 혐오를 하나의 **'부정'** 감정으로 묶고, 놀람, 중립, 행복 감정과 함께 총 4가지 감정 카테고리($M=4$)에 대한 통계를 도출합니다.
각 시기별 기사 수 차이(시행일 85개 vs 타 시기 각 2,800여 개)로 인한 왜곡을 방지하기 위해 **라플라스 스무딩(Laplace Smoothing, k=0.5)** 기법을 동일하게 적용하여 증감율을 계산합니다.

In [50]:
def analyze_emotion_changes_grouped(df_policy, k=0.5):
    """
    공포, 분노, 슬픔, 혐오 -> '부정'
    중립, 놀람 -> '중립'
    행복 -> '긍정'
    으로 매핑하여 시행일 기준 감정 비율의 증감율(%)을 계산합니다. (라플라스 스무딩 k=0.5, M=3)
    """
    periods = ['시행전', '시행일', '초기반응', '체감반응']
    emotions = ['부정', '중립', '긍정']
    
    # 날짜와 매핑 설정 (행복 -> 긍정, 놀람 -> 중립)
    df_policy = df_policy.copy()
    df_policy['그룹감정'] = df_policy['감정'].map(
        lambda x: '부정' if x in ['공포', '분노', '슬픔', '혐오'] else ('중립' if x in ['중립', '놀람'] else ('긍정' if x == '행복' else x))
    )
    
    # 1. 각 시기별 전체 기사 수 계산
    total_counts = {}
    for p in periods:
        total_counts[p] = len(df_policy[df_policy['시기'] == p])
        
    # 2. 각 시기별 그룹감정 빈도수 계산
    counts = {p: {e: 0 for e in emotions} for p in periods}
    for p in periods:
        df_sub = df_policy[df_policy['시기'] == p]
        for _, row in df_sub.iterrows():
            emo = row['그룹감정']
            if emo in emotions:
                counts[p][emo] += 1
                
    # 3. 라플라스 스무딩 비율 계산 (M = 3)
    M = len(emotions)
    smoothed_ratios = {p: {} for p in periods}
    for p in periods:
        total = total_counts[p]
        for e in emotions:
            count = counts[p][e]
            smoothed_ratios[p][e] = (count + k) / (total + k * M)
            
    # 4. 시행일(기준) 대비 증감율 계산
    changes = {p: {} for p in periods if p != '시행일'}
    for p in changes.keys():
        for e in emotions:
            r_p = smoothed_ratios[p][e]
            r_d = smoothed_ratios['시행일'][e]
            changes[p][e] = ((r_p - r_d) / r_d) * 100
            
    # 5. 결과 데이터프레임 빌드
    report_data = []
    for e in emotions:
        row = {'감정': e}
        row['시행일 비율(보정)'] = f"{counts['시행일'][e]}개 ({smoothed_ratios['시행일'][e]*100:.1f}%)"
        for p in ['시행전', '초기반응', '체감반응']:
            change_val = changes[p][e]
            raw_ratio = (counts[p][e] / total_counts[p]) * 100 if total_counts[p] > 0 else 0
            row[f"{p} (증감율)"] = f"{counts[p][e]}개 ({raw_ratio:.1f}%, {change_val:+.1f}%)"
        report_data.append(row)
        
    df_report = pd.DataFrame(report_data).set_index('감정')
    return df_report

# 실행
df_policy = df if 'df' in globals() else df_analyzed
df_change_report = analyze_emotion_changes_grouped(df_policy, k=0.5)

print("=========================================================================")
print(" [시행일 기준 감정 그룹화 비율 증감 분석 (부정 / 중립 / 긍정 3개 감정군)]")
print("=========================================================================")
print("※ 부정 = 공포+분노+슬픔+혐오 / 중립 = 중립+놀람 / 긍정 = 행복")
print("※ 증감율 = (비교시기 보정비율 - 시행일 보정비율) / 시행일 보정비율 * 100")
print("※ 포맷: 기사수 (실제비율%, 시행일 대비 보정 증감율%)\n")
import sys
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except AttributeError:
        pass
print(df_change_report.to_string())

 [시행일 기준 감정 그룹화 비율 증감 분석 (부정 / 중립 / 긍정 3개 감정군)]
※ 부정 = 공포+분노+슬픔+혐오 / 중립 = 중립+놀람 / 긍정 = 행복
※ 증감율 = (비교시기 보정비율 - 시행일 보정비율) / 시행일 보정비율 * 100
※ 포맷: 기사수 (실제비율%, 시행일 대비 보정 증감율%)

     시행일 비율(보정)              시행전 (증감율)             초기반응 (증감율)             체감반응 (증감율)
감정                                                                                  
부정  21개 (29.7%)   409개 (14.5%, -50.9%)   453개 (16.1%, -45.5%)   860개 (15.5%, -47.6%)
중립  49개 (68.3%)  2250개 (80.0%, +17.2%)  2210개 (78.8%, +15.3%)  4413개 (79.6%, +16.6%)
긍정    1개 (2.1%)   152개 (5.4%, +162.1%)   143개 (5.1%, +147.0%)   268개 (4.8%, +134.1%)


## 6단계: 시간 흐름에 따른 감정별 기사 수 시각화 (5일 단위)

전체 121일간의 뉴스 데이터를 일자별로 집계하고, 이를 5일 간격으로 합산(Resampling)하여 감정별 트렌드 변화를 선 그래프로 시각화합니다.
시행일(2025-06-28)은 파란색 점선으로 별도 표시하며, 완성된 그래프 이미지는 `images/emotion_trend.png` 파일로 자동 저장합니다.

In [53]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import os

# 윈도우 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

# 데이터 전처리 (놀람 -> 중립, 행복 -> 긍정)
df_viz = df.copy() if 'df' in globals() else df_analyzed.copy()
df_viz['날짜'] = pd.to_datetime(df_viz['날짜'])
df_viz['그룹감정'] = df_viz['감정'].map(
    lambda x: '부정' if x in ['공포', '분노', '슬픔', '혐오'] else ('중립' if x in ['중립', '놀람'] else ('긍정' if x == '행복' else x))
)

# 날짜 및 감정별 기사 수 집계
df_daily = df_viz.groupby(['날짜', '그룹감정']).size().unstack(fill_value=0)

# 121일 전체 날짜 재색인
all_dates = pd.date_range(start=df_daily.index.min(), end=df_daily.index.max(), freq='D')
df_daily = df_daily.reindex(all_dates, fill_value=0)

# 5일 단위로 '일평균' 기사 수 계산
df_5d = df_daily.resample('5D').mean()

# 시각화
fig, ax = plt.subplots(figsize=(14, 7))
colors = {'부정': '#e74c3c', '중립': '#95a5a6', '긍정': '#2ecc71'}

for col in ['부정', '중립', '긍정']:
    if col in df_5d.columns:
        ax.plot(df_5d.index, df_5d[col], marker='o', linewidth=2.5, color=colors[col], label=col)

# 시행일 세로 점선 표시 (2025-06-28)
effective_dt = pd.to_datetime('2025-06-28')
ax.axvline(x=effective_dt, color='#3498db', linestyle='--', linewidth=3, label='시행일 (2025-06-28)')

# x축 틱 간격을 5일 간격으로 설정
ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)

# 스타일 및 텍스트 설정
ax.set_title('6·27 부동산 대책 시간 흐름별 감정 수 추이 (5일 단위 일평균, 부정/중립/긍정)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('날짜 (연-월-일)', fontsize=12, labelpad=10)
ax.set_ylabel('일평균 뉴스 기사 수 (건/일)', fontsize=12, labelpad=10)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()

# images 폴더 생성 후 저장
os.makedirs("images", exist_ok=True)
image_path = "images/emotion_trend.png"
plt.savefig(image_path, dpi=150)
plt.close()
print(f"[Success] 시각화 그래프가 '{image_path}'에 성공적으로 저장되었습니다.")

[Success] 시각화 그래프가 'images/emotion_trend.png'에 성공적으로 저장되었습니다.


## 7단계: before/after 감정 비율 요약 CSV 내보내기

대책 시행 전(before)과 대책 시행 후(after) 기간에 대해 긍정, 중립, 부정의 비율(%)을 집계하여
`data/News_Scraping_retouch_summary.csv` 파일로 내보내어 분석 편의를 도모합니다.
- `before`: '시행전' 시기
- `after`: '시행일', '초기반응', '체감반응' 시기 합산

In [52]:
import pandas as pd
import os

# 1. 데이터 로드
df_summary = df.copy() if 'df' in globals() else df_analyzed.copy()

# 2. 감정 그룹화 (부정, 중립, 긍정)
df_summary['그룹감정'] = df_summary['감정'].map(
    lambda x: '부정' if x in ['공포', '분노', '슬픔', '혐오'] else ('중립' if x in ['중립', '놀람'] else ('긍정' if x == '행복' else x))
)

# 3. period 기간 분류 (before / after)
df_summary['period'] = df_summary['시기'].map(
    lambda x: 'before' if x == '시행전' else 'after'
)

# 4. period 및 그룹감정별 빈도수 피벗 테이블 생성
pivot_df = df_summary.groupby(['period', '그룹감정']).size().unstack(fill_value=0)

# 5. 백분율(%) 변환
ratio_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

# 6. 컬럼 순서 정렬 및 인덱스 리셋
ratio_df = ratio_df.reindex(columns=['긍정', '중립', '부정']).reset_index()

# 7. CSV 내보내기
os.makedirs("data", exist_ok=True)
output_csv_path = "data/News_Scraping_retouch_summary.csv"
ratio_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

print("============================================================")
print(" [요약 통계 CSV 생성 결과]")
print("============================================================")
print(ratio_df.to_string(index=False))
print(f"\n[Success] 요약 파일이 '{output_csv_path}'에 저장 완료되었습니다.")

 [요약 통계 CSV 생성 결과]
period       긍정        중립        부정
 after 4.894274 79.258731 15.846995
before 5.407328 80.042689 14.549982

[Success] 요약 파일이 'data/News_Scraping_retouch_summary.csv'에 저장 완료되었습니다.
